# 02주차 · 토큰화와 희소 문서 벡터

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 토큰화 규칙이 문서 표현을 바꾸는 과정을 설명한다.
- BoW와 TF-IDF를 비교한다.
- 희소 벡터 검색기를 임베딩 검색의 기준선으로 사용한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
documents = [
    "전남 청년 창업 지원 사업은 지역 정착을 지원한다",
    "목포 관광 스타트업에 사업화 자금을 지원한다",
    "고령자를 위한 복지관 이동 서비스를 확대한다",
    "섬 지역 주민에게 원격 의료 상담을 제공한다",
    "해양 관광 콘텐츠 기업의 창업을 지원한다",
]
documents


## TF-IDF

문서에 자주 나오지만 전체 말뭉치에서는 드문 단어에 큰 가중치를 준다.

$$\mathrm{tfidf}(t,d)=\mathrm{tf}(t,d)\log\frac{N+1}{\mathrm{df}(t)+1}$$

한국어는 조사·어미 때문에 띄어쓰기 토큰화가 완전하지 않다. 이번 실습에서는 단순 기준선을 먼저 만들고 형태소 분석은 확장 활동으로 둔다.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
X_count = count.fit_transform(documents)
count_df = pd.DataFrame(X_count.toarray(), columns=count.get_feature_names_out())
count_df


In [ ]:
tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
X = tfidf.fit_transform(documents)
query = "해양 관광 창업 지원"
q = tfidf.transform([query])
scores = (X @ q.T).toarray().ravel()
result = pd.DataFrame({"문서": documents, "점수": scores}).sort_values("점수", ascending=False)
result


## 학생 활동

- 질의어를 세 번 바꾸고 상위 문서가 바뀌는 이유를 토큰 단위로 설명하라.
- `ngram_range=(1, 2)`를 사용했을 때 결과를 비교하라.
- TF-IDF가 동의어를 잘 찾지 못하는 실패 사례를 한 개 만들라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
